In [5]:
import pymupdf4llm

In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter,RecursiveCharacterTextSplitter,Language
from langchain_core.documents import Document

In [7]:
from pathlib import Path
# Path containing your 6 policy PDFs
DATA_DIR = Path("data/policies")
pdf_files = list(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF(s) to process.\n")

# Step 1: Define Markdown Header hierarchy
headers_to_split_on = [("#", "Header 1"),("##", "Header 2"),("###", "Header 3"),]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on,
                                               strip_headers=False  # Keeps section titles inside the text
                                               )

# Step 2: Define Markdown-aware character splitter (large chunk size to protect tables)
text_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.MARKDOWN,
                                                             chunk_size=1200,      # Keeps multi-line tables intact
                                                             chunk_overlap=150
                                                             )

Found 5 PDF(s) to process.



In [8]:
all_documents = []

# Process each PDF file step-by-step
for idx, pdf_path in enumerate(pdf_files, start=1):
    insurer_name = pdf_path.stem.replace("_", " ").title()
    print(f"[{idx}/{len(pdf_files)}] Parsing: {pdf_path.name}...")

    # Extract Markdown page-by-page to retain accurate page numbers
    page_data = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)

    for page_dict in page_data:
        page_num = page_dict["metadata"]["page_number"]  # 1-based page number
        page_text = page_dict["text"]

        if not page_text.strip():
            continue

        # A. Split by Markdown headers first
        header_splits = markdown_splitter.split_text(page_text)

        # B. Sub-split large text blocks
        final_splits = text_splitter.split_documents(header_splits)

        # C. Attach file and page metadata
        for doc in final_splits:
            doc.metadata.update({
                "source": pdf_path.name,
                "insurer": insurer_name,
                "page": page_num
            })
            all_documents.append(doc)

print(f"\n✅ Processing Complete! Total Chunks Extracted: {len(all_documents)}")

[1/5] Parsing: Star_Comprehensive_Insurance_Policy.pdf...
[2/5] Parsing: Star_Diabetes_Safe_Insurance_Policy.pdf...
[3/5] Parsing: Star_Senior_Citizens_Red_Carpet_Health_Insurance_Policy.pdf...
[4/5] Parsing: Star_Women_Care_Insurance_Policy.pdf...
[5/5] Parsing: Star_Young_Extra_Protect_Add_On.pdf...

✅ Processing Complete! Total Chunks Extracted: 291


In [9]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Aswin\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aswin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8105.62it/s]


In [ ]:
from langchain_community.vectorstores import FAISS

print("Generating vector embeddings and saving local FAISS index...")

# Vectorize all chunks and store in FAISS database
vectorstore = FAISS.from_documents(all_documents, embeddings)

# Save index to local folder
vectorstore.save_local("faiss_index")

print("'faiss_index/' directory created with index.faiss and index.pkl!")

Generating vector embeddings and saving local FAISS index...
🎉 SUCCESS! 'faiss_index/' directory created with index.faiss and index.pkl!


In [ ]:
! pip install faiss-cpu